In [2]:
#rnn模型介绍
#循环神经网络（Recurrent Neural Network，RNN）是一种用于处理序列数据的神经网络模型。RNN模型通过在时间步之间共享参数来捕捉序列中的依赖关系，使其能够处理变长的输入和输出序列。RNN模型在自然语言处理、语音识别、时间序列预测等领域有广泛的应用。
#RNN模型的基本结构包括输入层、隐藏层和输出层。输入层接受序列数据，隐藏层通过循环连接来捕捉序列中的依赖关系
#输出层根据隐藏层的输出生成最终的预测结果。RNN模型的训练通常使用反向传播算法，通过最小化损失函数来更新模型参数。

#rnn模型的分类
#分为N vs N模型、N vs 1模型和N vs M模型三种类型。
#下面介绍的是N vs N模型，也称为序列到序列模型（Sequence-to-Sequence Model）。在N vs N模型中，输入和输出都是序列，且输入和输出的长度可以不同。该模型通常由一个编码器（Encoder）和一个解码器（Decoder）组成。编码器将输入序列编码成一个固定长度的向量表示，解码器根据这个向量表示生成输出序列。N vs N模型在机器翻译、文本生成等任务中有广泛的应用。

#传统rnn模型结构
#传统RNN模型的结构包括输入层、隐藏层和输出层。输入层接受序列数据，隐藏层通过循环连接来捕捉序列中的依赖关系，输出层根据隐藏层的输出生成最终的预测结果。传统RNN模型在处理长序列时可能会遇到梯度消失或梯度爆炸的问题，因此在实际应用中，通常会使用改进的RNN模型，如长短期记忆网络（LSTM）或门控循环单元（GRU）来解决这些问题。

#RNN的计算过程
#假设输入序列为x=(x1,x2,x3,...,xt)，其中xt是第t个时间步的输入。
#假设隐藏状态为h=(h1,h2,h3,...,ht)，其中ht是第t个时间步的隐藏状态。
#假设输出序列为y=(y1,y2,y3,...,yt)，其中yt是第t个时间步的输出。
#假设RNN模型的参数为Wih、Whh、Why和bh、bh和by，其中Wih是输入到隐藏状态的权重矩阵，Whh是隐藏状态到隐藏状态的权重矩阵，Why是隐藏状态到输出的权重矩阵，bh是隐藏状态的偏置向量，by是输出的偏置向量。
#则RNN模型的计算过程如下：
#ht=σ(Wihxt+Whhht−1+bh)
#yt=Whyht+by
#其中，σ是激活函数，例如sigmoid函数或tanh函数。
#ht是第t个时间步的隐藏状态，依赖于第t-1个时间步的隐藏状态和第t个时间步的输入。
#yt是第t个时间步的输出，依赖于第t个时间步的隐藏状态。

import torch.nn as nn
import torch



In [10]:
##rnn层 api
#rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
#input_size:输入特征的维度，一般为词嵌入层的输出维度
#hidden_size:隐藏层的维度，表示每个时间步输出的特征数量
#num_layers:rnn层的层数
#batch_first:如果为True,则输入和输出张量的形状为(batch_size,seq_len,input_size)
#否则为(seq_len,batch_size,input_size)

input=torch.randn(32,2,128) #各维度分别为批次大小、序列长度和输入特征维度，批次大小是每次输入模型的样本数量，序列长度是每个样本的时间步数，输入特征维度是每个时间步输入的特征数量,也是词向量维数
input_size=128
hidden_size=256
num_layers=1

h0=torch.randn(num_layers,32,hidden_size) #各维度分别为层数、批次大小和隐藏层维度，层数是rnn层的层数，批次大小是每次输入模型的样本数量，隐藏层维度是每个时间步输出的特征数量
rnn=nn.RNN(input_size, hidden_size, num_layers,batch_first=True)
output,hidden=rnn(input,h0) #output是RNN层的输出，hidden是RNN层的隐藏状态
print(output) #输出张量的形状为(32,5,256)，各维度分别为批次大小、序列长度和隐藏层维度
print(hidden) #隐藏状态的形状为(1,32,256)，各维度分别为层数、批次大小和隐藏层维度

"""
    多层RNN (3层) 处理序列 "A B C D":

    时间轴:     t=0    t=1    t=2    t=3
    ------------------------------------
    输入:       A      B      C      D
                ↓      ↓      ↓      ↓
    第1层:     h1₀    h1₁    h1₂    h1₃
                ↓      ↓      ↓      ↓
    第2层:     h2₀    h2₁    h2₂    h2₃
                ↓      ↓      ↓      ↓
    第3层:     h3₀    h3₁    h3₂    h3₃
               ↓      ↓      ↓      ↓

    output = [h3₀, h3₁, h3₂, h3₃]  ← 每个时间步的最后一层状态

    hn = [h1₃, h2₃, h3₃]  ← 最后一个时间步的所有层状态

    关系:
    - output包含每个时间步(t)的最后一层状态
    - hn包含每个层(l)的最后一个时间步状态
    - output[:, -1, :] = hn[-1, :, :] (都是h3₃)
    """



tensor([[[ 0.3683, -0.8743, -0.6375,  ..., -0.2380, -0.4921, -0.1326],
         [-0.2895, -0.3709, -0.4436,  ..., -0.5072,  0.3297,  0.2208]],

        [[-0.3055, -0.7794, -0.6450,  ...,  0.7034, -0.9709, -0.4452],
         [-0.1913,  0.7446,  0.2953,  ..., -0.0394,  0.3960, -0.8055]],

        [[-0.2349, -0.7083,  0.0675,  ..., -0.8265,  0.1221, -0.0109],
         [ 0.0046,  0.1298, -0.5480,  ..., -0.1367, -0.2546,  0.6494]],

        ...,

        [[ 0.4585, -0.1037,  0.3282,  ..., -0.1732, -0.4065, -0.1583],
         [-0.4856,  0.1481, -0.3489,  ...,  0.5033, -0.4068,  0.1937]],

        [[-0.3457,  0.1290,  0.5015,  ..., -0.5783, -0.4733,  0.4186],
         [-0.0820, -0.5312, -0.3805,  ...,  0.3383, -0.7901,  0.1245]],

        [[-0.7890,  0.4027,  0.3020,  ...,  0.4099,  0.5734,  0.6122],
         [-0.5918, -0.1304, -0.4960,  ...,  0.2528,  0.1313,  0.4546]]],
       grad_fn=<TransposeBackward1>)
tensor([[[-0.2895, -0.3709, -0.4436,  ..., -0.5072,  0.3297,  0.2208],
         [-0.1

In [ ]:
#传统rnn缺点
#1.梯度消失和梯度爆炸问题：在训练过程中，RNN模型可能会遇到梯度消失或梯度爆炸的问题，导致模型难以学习长序列中的依赖关系。
#2.计算效率低：由于RNN模型需要逐步处理序列数据，因此在处理长序列时计算效率较低。
#3.难以捕捉长距离依赖关系：RNN模型在处理长序列时，难以捕捉长距离依赖关系，导致模型性能下降。





In [ ]:
#LSTM模型介绍
#长短期记忆网络（Long Short-Term Memory, LSTM）是一种改进的循环神经网络（RNN）模型，旨在解决传统RNN模型中的梯度消失和梯度爆炸问题。LSTM模型通过引入门控机制来控制信息的流动，使得模型能够更好地捕捉长距离依赖关系。LSTM模型在自然语言处理、语音识别、时间序列预测等领域有广泛的应用。
#LSTM模型的结构包括输入门、遗忘门、输出门和记忆单元。输入门控制当前输入信息的流入，遗忘门控制之前记忆信息的保留，输出门控制当前记忆信息的输出，记忆单元用于存储长期依赖信息。LSTM模型通过这些门控机制来有效地捕捉序列中的依赖关系，从而提高模型的性能。

## 遗忘门公式
# f_t = σ(W_f · [h_{t-1}, x_t] + b_f)

# 在细胞状态更新中的应用
# C_t = f_t ⊙ C_{t-1} + ...  # 选择性保留旧信息

# 输入门公式
# i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
#候选值公式
# C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C)
# 在细胞状态更新中的应用
# C_t = ... + i_t ⊙ C̃_t  # 选择性添加新信息
#细胞状态更新公式
# C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t
#输出门公式
# o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
#隐藏状态更新公式
# h_t = o_t ⊙ tanh(C_t)

#公式解释
#1. 遗忘门 (f_t)：控制之前记忆信息的保留程度。通过sigmoid函数输出0到1之间的值，0表示完全遗忘，1表示完全保留。
#2. 输入门 (i_t)：控制当前输入信息的流入程度。通过sigmoid函数输出0到1之间的值，0表示完全不接受，1表示完全接受。
#3. 候选值 (C̃_t)：通过tanh函数生成当前输入和之前隐藏状态的候选记忆信息，值范围在-1到1之间。
#4. 细胞状态更新 (C_t)：通过遗忘门和输入门的控制，选择性地保留之前的记忆信息和添加新的候选记忆信息，从而更新细胞状态。
#5. 输出门 (o_t)：控制当前记忆信息的输出程度。通过sigmoid函数输出0到1之间的值，0表示完全不输出，1表示完全输出。
#6. 隐藏状态更新 (h_t)：通过输出门的控制，将当前记忆信息经过tanh函数处理后输出作为当前时间步的隐藏状态。

#BI-LSTM模型介绍
#双向长短期记忆网络（Bidirectional LSTM, BI-LSTM）是一种改进的LSTM模型，旨在捕捉序列数据中的双向依赖关系。BI-LSTM模型通过在正向和反向两个方向上同时处理序列数据，使得模型能够更好地理解上下文信息。BI-LSTM模型在自然语言处理、语音识别、时间序列预测等领域有广泛的应用。
#BI-LSTM模型的结构包括两个LSTM层，一个用于正向处理序列数据，另一个用于反向处理序列数据。正向LSTM层从序列的开始到结束处理数据，反向LSTM层从序列的结束到开始处理数据。BI-LSTM模型通过将正向和反向LSTM层的输出进行拼接或加权平均来生成最终的输出，从而捕捉序列中的双向依赖关系，提高模型的性能。
#举例说明
#假设输入序列为 "I love machine learning"，BI-LSTM模型将同时处理正向序列 "I → love → machine → learning" 和反向序列 "learning → machine → love → I"。正向LSTM层将捕捉到 "I" 和 "love" 之间的依赖关系，而反向LSTM层将捕捉到 "learning" 和 "machine" 之间的依赖关系。最终，BI-LSTM模型将结合正向和反向的信息来生成更丰富的上下文表示，从而提高对输入序列的理解和预测能力。






In [13]:
#使用pytorch进行LSTM模型实现
#api使用和rnn类似
batch_size=32
seq_len=10
input_size=128
num_layers=1
input=torch.randn(batch_size,seq_len,input_size)
h0=torch.randn(num_layers,batch_size,hidden_size)
c0=torch.randn(num_layers,batch_size,hidden_size) #LSTM模型需要初始化两个隐藏状态，分别是h0和c0
lstm=nn.LSTM(input_size,hidden_size,num_layers,batch_first=True)
output,(hn,cn)=lstm(input,(h0,c0))
print(output)

print(hn)
print(cn)
print(output.shape) #输出张量的形状为(32,10,256)，各维度分别为批次大小、序列长度和隐藏层维度
print(hn.shape) #隐藏状态的形状为(1,32,256)，各维度分别为层数、批次大小和隐藏层维度
print(cn.shape) #细胞状态的形状为(1,32,256)，各维度分别为层数、批次大小和隐藏层维度

#output和hn的关系
#output包含每个时间步(t)的最后一层状态，而hn包含每个层(l)的最后一个时间步状态。在单层LSTM中，output[:, -1, :] 和 hn[-1, :, :] 都表示最后一个时间步的隐藏状态，因此它们是相等的。在多层LSTM中，output[:, -1, :] 表示最后一个时间步的最后一层状态，而 hn[-1, :, :] 表示最后一个时间步的所有层状态，因此它们不一定相等。




tensor([[[ 1.7387e-01, -2.8798e-01,  9.1099e-02,  ..., -2.0078e-02,
           1.8881e-01,  5.1960e-02],
         [ 9.4497e-02,  3.8296e-02,  3.8115e-02,  ..., -4.4623e-02,
           1.1963e-01, -6.5846e-02],
         [ 1.3280e-01, -2.5427e-02,  6.9141e-02,  ..., -9.6909e-02,
           1.2166e-01, -7.3106e-02],
         ...,
         [ 1.1674e-01,  1.7233e-01,  1.6654e-02,  ..., -2.0408e-02,
          -1.1767e-01,  3.1434e-02],
         [ 1.9303e-01,  1.1842e-01,  7.0123e-02,  ..., -9.4396e-02,
           2.8865e-02, -5.6005e-02],
         [ 1.4965e-01, -3.7661e-02, -1.9698e-02,  ...,  1.0469e-01,
          -1.1391e-01, -4.8085e-02]],

        [[-1.0837e-01,  1.4296e-01,  1.4386e-01,  ...,  3.5913e-01,
          -4.4712e-02, -1.7719e-01],
         [-8.7810e-02,  1.3461e-02,  1.9939e-01,  ...,  2.4271e-01,
          -1.1935e-01, -7.2132e-02],
         [-1.5220e-01,  2.8524e-01, -3.1415e-02,  ...,  1.3779e-01,
          -3.5653e-02, -1.0849e-01],
         ...,
         [-3.4338e-02,  2

In [ ]:
#GRU模型介绍
#门控循环单元（Gated Recurrent Unit, GRU）是一种改进的循环神经网络（RNN）模型，旨在解决传统RNN模型中的梯度消失和梯度爆炸问题。GRU模型通过引入更新门和重置门来控制信息的流动，使得模型能够更好地捕捉长距离依赖关系。GRU模型在自然语言处理、语音识别、时间序列预测等领域有广泛的应用。
#GRU模型的结构包括更新门、重置门和候选隐藏状态。更新门控制当前隐藏状态的更新程度，重置门控制之前隐藏状态的重置程度，候选隐藏状态用于生成当前时间步的隐藏状态。GRU模型通过这些门控机制来有效地捕捉序列中的依赖关系，从而提高模型的性能。

#GRU和LSTM的区别
#1. 门控机制：GRU模型只有两个门（更新门和重置门），而LSTM模型有三个门（输入门、遗忘门和输出门）。
#2. 隐藏状态更新：GRU模型直接使用候选隐藏状态来更新隐藏状态，而LSTM模型通过输入门和遗忘门的控制来更新细胞状态和隐藏状态。
#3. 模型复杂度：由于GRU模型的门控机制较为简单，因此相对于LSTM模型来说，GRU模型的参数较少，计算效率较高，适用于较小的数据集和较短的序列数据。

#重置门公式
# r_t = σ(W_r · [h_{t-1}, x_t] + b_r)
#更新门公式
# z_t = σ(W_z · [h_{t-1}, x_t] + b_z)
#候选隐藏状态公式
# h̃_t = tanh(W_h · [r_t ⊙ h_{t-1}, x_t] + b_h)
#隐藏状态更新公式
# h_t = (1 - z_t) ⊙ h_{t-1}+ z_t ⊙ h̃_t

#公式解释
#1. 重置门 (r_t)：控制之前隐藏状态的重置程度。通过sigmoid函数输出0到1之间的值，0表示完全重置，1表示完全保留。
#2. 更新门 (z_t)：控制当前隐藏状态的更新程度。通过sigmoid函数输出0到1之间的值，0表示完全不更新，1表示完全更新。
#3. 候选隐藏状态 (h̃_t)：通过tanh函数生成当前输入和之前隐藏状态的候选隐藏状态，值范围在-1到1之间。重置门控制之前隐藏状态的影响程度。
#4. 隐藏状态更新 (h_t)：通过更新门的控制，选择性地保留之前的隐藏状态和添加新的候选隐藏状态，从而更新当前时间步的隐藏状态。
#与LSTM模型对比
#1. GRU模型只有两个门（更新门和重置门），而LSTM模型有三个门（输入门、遗忘门和输出门）。
#2. GRU模型直接使用候选隐藏状态来更新隐藏状态，而LSTM模型通过输入门和遗忘门的控制来更新细胞状态和隐藏状态。
#3.GRU模型没有细胞状态，因此相对于LSTM模型来说，GRU模型的参数较少，计算效率较高，适用于较小的数据集和较短的序列数据。







In [14]:
#GRU代码的实现:与LSTM类似，但不需要初始化细胞状态c0
input=torch.randn(batch_size,seq_len,input_size)
h0=torch.randn(num_layers,batch_size,hidden_size)
gru=nn.GRU(input_size,hidden_size,num_layers,batch_first=True)
output,hn=gru(input,h0)
print(output)
print(hn)
print(output.shape)
print(hn.shape)

tensor([[[ 1.2496e-01, -5.5778e-01, -6.9879e-01,  ..., -2.8282e-01,
          -3.6794e-01,  1.3708e+00],
         [-2.7117e-02, -7.8033e-02, -5.7098e-01,  ...,  8.9954e-02,
          -3.5502e-01,  5.0078e-01],
         [-6.6333e-02, -8.2577e-03, -2.1475e-01,  ...,  8.0904e-02,
          -5.2889e-02, -1.4069e-03],
         ...,
         [ 2.4849e-02,  2.0311e-01, -4.4607e-01,  ..., -6.3575e-02,
          -1.2806e-01,  2.4764e-01],
         [-2.7554e-01,  2.6083e-01,  1.7080e-02,  ..., -1.1745e-01,
          -1.0913e-01,  7.0530e-02],
         [-2.4093e-01, -7.3883e-02, -1.6273e-01,  ..., -1.7728e-01,
          -6.1769e-02, -6.1323e-02]],

        [[-1.2603e-01,  1.1033e+00,  1.1390e-01,  ..., -1.1258e-01,
           8.9003e-01, -8.3501e-01],
         [-5.7505e-02,  3.6567e-01,  1.7297e-01,  ..., -3.4227e-01,
           1.5785e-01, -1.2973e-01],
         [-1.3290e-01,  5.4406e-01, -1.6471e-01,  ..., -1.5944e-01,
           2.0502e-01,  1.1286e-01],
         ...,
         [-1.8691e-01, -5

In [41]:
#rnn案例：人名分类器
#todo:使用rnn,lstm,gru三种模型实现人名分类器，并比较它们的性能。

#数据处理
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import time

names=pd.read_csv('./data/name_classfication.txt',sep='\t')
print(names.head())
names.columns=['name','class']
names.head()
x=names['name']
y=names['class']
class_list=list(set(y))
print(class_list)
class_idx={c:i for i,c in enumerate(class_list)}
print(class_idx)
y=[class_idx[c] for c in y]
print(y)
x_unique=list(set(x))
print(len(x_unique))
print(len(x))
name_idx={n:i for i,n in enumerate(x_unique)}
idx=[name_idx[n] for n in x]

#将每个名字分解为字符序列
all_chars=set()
for name in names['name']:
    all_chars.update(list(name))
all_chars=list(all_chars)
char_to_idx={c:i for i,c in enumerate(all_chars)}
print(f"字符表大小: {len(all_chars)}")

#将名字转换为字符索引序列
name_sequences=[]
max_len=0
for name in names['name']:
    seq=[char_to_idx[c] for c in name]
    name_sequences.append(seq)
    max_len=max(max_len,len(seq))
print(f"最大名字长度: {max_len}")

#进行padding
from torch.nn.utils.rnn import pad_sequence
padded_sequences=[]
for seq in name_sequences:
    padded_seq=seq+[0]*(max_len-len(seq))
    padded_sequences.append(padded_seq)
x=torch.tensor(padded_sequences)
print(f"x的形状: {x.shape}")


name_dataset=TensorDataset(x,torch.tensor(y))
name_dataloader=DataLoader(name_dataset,batch_size=32,shuffle=True)
batch_size=32
num_layers=1
hidden_size=256

#搭建模型
class NameClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding=nn.Embedding(len(all_chars),128)
        self.rnn=nn.RNN(input_size=128,hidden_size=hidden_size,num_layers=num_layers,batch_first=True)
        self.fc1=nn.Linear(hidden_size,len(class_idx))
    def forward(self,x,h0):
        x=self.embedding(x)
        output,hn=self.rnn(x,h0)
        output=self.fc1(output[:,-1,:]) #取最后一个时间步的输出作为分类器的输入
        return output,hn
    def init_hidden(self,Batch_size):
        return torch.zeros(num_layers,Batch_size,hidden_size)


#模型训练
model_rnn=NameClassifier()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model_rnn.parameters(),lr=0.001)
epochs=30
for epoch in range(epochs):
    model_rnn.train()
    total_samples,total_loss,total_correct,start=0,0,0,time.time()
    for x,y in name_dataloader:
        optimizer.zero_grad()
        h0=model_rnn.init_hidden(x.size(0))
        output,hn=model_rnn(x,h0)
        loss=criterion(output,y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()*x.size(0)
        total_samples+=x.size(0)
        total_correct+=(torch.argmax(output,dim=1)==y).sum()
    end=time.time()
    print(f"epoch:{epoch},loss:{total_loss/total_samples:.4f},accuracy:{total_correct/total_samples:.4f},time:{end-start:.2f}s")



            Abl  Czech
0         Adsit  Czech
1        Ajdrna  Czech
2           Alt  Czech
3  Antonowitsch  Czech
4    Antonowitz  Czech
['French', 'English', 'Spanish', 'Portuguese', 'Korean', 'Russian', 'Polish', 'Irish', 'Dutch', 'Scottish', 'Arabic', 'Vietnamese', 'Italian', 'Czech', 'German', 'Chinese', 'Japanese', 'Greek']
{'French': 0, 'English': 1, 'Spanish': 2, 'Portuguese': 3, 'Korean': 4, 'Russian': 5, 'Polish': 6, 'Irish': 7, 'Dutch': 8, 'Scottish': 9, 'Arabic': 10, 'Vietnamese': 11, 'Italian': 12, 'Czech': 13, 'German': 14, 'Chinese': 15, 'Japanese': 16, 'Greek': 17}
[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,